Install PLY

In [ ]:
!pip install ply

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 674.4 kB/s eta 0:00:00


Import Libraries

In [ ]:
import sys
import ply.lex as lex
import ply.yacc as yacc

# Fix for Google Colab
sys.modules['__main__'].__file__ = 'postfix.py'

Define Lexer

In [ ]:
tokens = (
    'NUMBER',
    'PLUS',
    'MINUS',
    'TIMES',
    'DIVIDE'
)

t_PLUS = r'\+'
t_MINUS = r'-'
t_TIMES = r'\*'
t_DIVIDE = r'/'

def t_NUMBER(t):
    r'\d+(\.\d+)?'

    if '.' in t.value:
        t.value = float(t.value)
    else:
        t.value = int(t.value)

    return t

t_ignore = ' \t'

def t_newline(t):
    r'\n+'
    t.lexer.lineno += len(t.value)

def t_error(t):
    print(f"Illegal character '{t.value[0]}' at line {t.lineno}, position {t.lexpos}")
    t.lexer.skip(1)

lexer = lex.lex()

Define Parser

In [ ]:
def p_expression_number(p):
    'expression : NUMBER'
    p[0] = p[1]


def p_expression_add(p):
    'expression : expression expression PLUS'
    p[0] = p[1] + p[2]


def p_expression_subtract(p):
    'expression : expression expression MINUS'
    p[0] = p[1] - p[2]


def p_expression_multiply(p):
    'expression : expression expression TIMES'
    p[0] = p[1] * p[2]


def p_expression_divide(p):
    'expression : expression expression DIVIDE'

    if p[2] == 0:
        print("Error: Division by zero")
        p[0] = None
    else:
        p[0] = p[1] / p[2]


def p_error(p):
    if p is None:
        print("Syntax error at EOF")
    else:
        print(f"Syntax error at '{p.value}'")


parser = yacc.yacc(write_tables=False, debug=False)

Test 2 3 +

In [ ]:
expression = "2 3 +"

lexer.lineno = 1
result = parser.parse(expression, lexer=lexer)

if result is not None:
    print("Result:", result)

Result: 5


Test 2 3 + 4 *

In [ ]:
expression = "2 3 + 4 *"

lexer.lineno = 1
result = parser.parse(expression, lexer=lexer)

if result is not None:
    print("Result:", result)

Result: 20


Test Floating-Point Operands

In [ ]:
expression = "2.5 3.5 +"

lexer.lineno = 1
result = parser.parse(expression, lexer=lexer)

if result is not None:
    print("Result:", result)

Result: 6.0


Test All Operators

In [ ]:
test_expressions = [
    "2 3 +",
    "5 2 -",
    "4 3 *",
    "10 2 /",
    "2 3 + 4 *",
    "10 2 / 3 +",
    "2.5 3.5 +"
]

for expression in test_expressions:
    print("\nInput:", expression)

    lexer.lineno = 1
    result = parser.parse(expression, lexer=lexer)

    if result is not None:
        print("Result:", result)


Input: 2 3 +
Result: 5

Input: 5 2 -
Result: 3

Input: 4 3 *
Result: 12

Input: 10 2 /
Result: 5.0

Input: 2 3 + 4 *
Result: 20

Input: 10 2 / 3 +
Result: 8.0

Input: 2.5 3.5 +
Result: 6.0


Test Invalid Expression

In [ ]:
expression = "2 +"

lexer.lineno = 1
parser.parse(expression, lexer=lexer)

Syntax error at '+'


Test Division by Zero

In [ ]:
expression = "10 0 /"

lexer.lineno = 1
result = parser.parse(expression, lexer=lexer)

Error: Division by zero


Test Illegal Character

In [ ]:
expression = "2 3 @"

lexer.lineno = 1
result = parser.parse(expression, lexer=lexer)

Illegal character '@' at line 1, position 4
Syntax error at EOF


# Algorithm

1. Read the postfix expression.
2. The lexical analyzer scans the input using PLY Lex.
3. Identify integer and floating-point numbers as NUMBER tokens.
4. Identify +, -, * and / as operator tokens.
5. Pass the generated tokens to the parser.
6. The parser checks the postfix expression using the grammar rules.
7. For a NUMBER, return its value.
8. For a binary operator, take the two preceding expressions and perform the corresponding arithmetic operation.
9. Handle division by zero as an error.
10. If the expression is invalid, display an appropriate syntax error.
11. If the expression is valid, display the calculated result.

# Grammar

expression → NUMBER

expression → expression expression +

expression → expression expression -

expression → expression expression *

expression → expression expression /

Examples:

2 3 +
→ 5

2 3 + 4 *
→ (2 + 3) * 4
→ 20

# Working

For the postfix expression:

2 3 + 4 *

The lexer generates:

NUMBER(2)
NUMBER(3)
PLUS(+)
NUMBER(4)
TIMES(*)

The parser evaluates the postfix expression:

2 3 +
→ 5

5 4 *
→ 20

Therefore:

Output → 20